# Transformation inputs: demo on the running example

Concept: `overview/concepts/Transformations.md`, section "Deriving transformation events and consumed/produced handling units", options 1-6 (outside this repo).
Code: `ocelescope_module_location_based_ocpm.derivation.transformations` (the options) and `derivation.inputs.resolve_transformation_inputs` (reads what an option needs from the log); the configuration models are in `derivation/config.py`.
This notebook only imports and calls these functions. Further detail: the docstrings and `docs/resources/transformation-inputs.md`.

What the notebook does:
1. loads the running example and its configuration and resolves the inputs (the default: option 5),
2. runs every option on the running example with `resolve_transformations`,
3. shows the assumption checks AS-T1 to AS-T4,
4. saves the resolved roles to `data/running_example/derived/transformation_roles.csv`.

In [ ]:
import json
from pathlib import Path

import polars as pl
from ocelescope import OCEL

from ocelescope_module_location_based_ocpm.derivation.config import (
    DerivationConfig,
    RolesFromFirstLast,
    RolesFromFrequency,
    RolesFromObjectSets,
    RolesFromParents,
    RolesFromQualifiers,
    RolesFromTable,
)
from ocelescope_module_location_based_ocpm.derivation.inputs import resolve_inputs
from ocelescope_module_location_based_ocpm.derivation.transformations import (
    activity_frequencies,
    apply_as_t1,
    check_assumptions,
    first_last_events,
    resolve_transformations,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"
INPUTS = EXAMPLE / "inputs"
DERIVED = EXAMPLE / "derived"

pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250);

## 1. Inputs

The running example configuration has transformation events (activities cast slab, roll, cut) and no "Consumed and produced handling units" input, so the default applies: option 5 (first and last timestamps).
`resolve_inputs` resolves everything; `resolved.roles` has one row per consumed or produced handling unit (columns event, hu, role), and the resolved transformation events T are the events of the relations with `is_transformation`.

In [ ]:
ocel = OCEL.read(EXAMPLE / "running_example.sqlite")
config = DerivationConfig.model_validate(json.loads((EXAMPLE / "configuration.json").read_text()))
tables = {"locations.csv": pl.read_csv(INPUTS / "locations.csv", infer_schema=False)}
resolved = resolve_inputs(ocel, config, tables)
resolved.notices, resolved.errors

In [ ]:
resolved.roles.sort("event", "role", "hu")

The options below are called on frames, so they need the hu-event relations `e2hu` (columns event, hu, object_type, time, activity), the handling units HU, all event ids, the candidates (the transformation-events input) and the processing operations for the tie-break.

In [ ]:
e2hu = resolved.relations.select("event", "hu", "object_type", "time", "activity")
handling_units = e2hu["hu"].unique()
events = resolved.events["event"]
candidates = pl.read_csv(INPUTS / "transformation_events.csv")["event"]
processing = resolved.events.filter(pl.col("activity") == "label")["event"]  # the explicit selection only
candidates.to_list(), processing.to_list()

## 2. First and last event per handling unit

`first_last_events(e2hu, processing)`: per handling unit its first event $e^{first}(hu)$ and last event $e^{last}(hu)$.
Ties at the same timestamp: processing operations first, then by event id ($\prec$).
Only the explicitly selected processing operations are used (not "+ all transformation events"), because otherwise option 6 would depend on its own result.
Options 3-6 use it.

In [ ]:
first_last_events(e2hu, processing)

## 3. The options

`resolve_transformations(option, candidates, e2hu, handling_units, events, processing, e2o, parents, tables, object_sets)` returns `ResolvedTransformations` with
- `events`: T after the AS-T1 filter,
- `roles`: the consumed and produced handling units (event, hu, role),
- `notices`: dropped events (AS-T1), violations of AS-T2 to AS-T4, and "transformation_events_ignored" if the option derives T although candidates were given.

Which options need the transformation-events input:

| Option | Config model | T | Needs |
|---|---|---|---|
| 1 e2o qualifiers | `RolesFromQualifiers` | derived (input ignored) | consumed and produced qualifiers |
| 2 table | `RolesFromTable` | candidates, or the table's events | table with event, object and role columns |
| 3 parent relations | `RolesFromParents` | candidates (required) | o2o qualifiers and their direction |
| 4 produced/consumed objects | `RolesFromObjectSets` | candidates (required) | object sets (attribute values or table), final or partial consumption |
| 5 first/last timestamps | `RolesFromFirstLast` | candidates (required) | nothing; the default |
| 6 frequency | `RolesFromFrequency` | derived (input ignored) | threshold $\tau$ (default 0.5) |

Without either input, T = ∅.

In [ ]:
def show(result):
    """Print T and the notices and return the roles."""
    print("T =", result.events.sort().to_list())
    for notice in result.notices:
        print(f"{notice.code}: {notice.message} {notice.ids}")
    return result.roles.sort("event", "role", "hu")

### Option 5: first and last timestamps (default)

prod(e) = handling units whose first event is e, cons(e) = handling units whose last event is e, for e among the candidates.

In [ ]:
show(resolve_transformations(RolesFromFirstLast(), candidates, e2hu, handling_units, events, processing))

### Option 1: e2o qualifiers

The running example relates consumed and produced objects with the qualifiers `consumed` and `produced`. T is derived from them; passing candidates as well gives the notice "transformation_events_ignored".

In [ ]:
e2o = ocel.e2o.pl.select("ocel:eid", "ocel:oid", "ocel:qualifier").collect().rename(
    {"ocel:eid": "event", "ocel:oid": "hu", "ocel:qualifier": "qualifier"}
)
option = RolesFromQualifiers(consumed_qualifiers=["consumed"], produced_qualifiers=["produced"])
show(resolve_transformations(option, None, e2hu, handling_units, events, processing, e2o=e2o))

### Option 2: table

`inputs/transformation_roles.csv` has the columns event, hu, role with the values `consumed` and `produced` (the defaults of `consumed_values` and `produced_values`).
Without candidates, T is the table's events; with candidates, AS-T1 drops candidates without roles (here e4 is added as a candidate to show the notice).

In [ ]:
table_option = RolesFromTable(table="transformation_roles.csv", event_column="event", object_column="hu", role_column="role")
role_tables = {"transformation_roles.csv": pl.read_csv(INPUTS / "transformation_roles.csv")}
show(resolve_transformations(table_option, None, e2hu, handling_units, events, processing, tables=role_tables))

In [ ]:
with_e4 = pl.concat([candidates, pl.Series("event", ["e4"])])
show(resolve_transformations(table_option, with_e4, e2hu, handling_units, events, processing, tables=role_tables))

### Option 3: parent relations

From o2o relations child → parent: prod(e) = handling units with a parent whose first event is e, cons(e) = parents of those whose last event is e (see the note for the exact rule, "not earlier than").
The running example log has no o2o relations, so the parent relations are given as a frame here (in the module they come from the o2o qualifiers, `direction` saying whether the qualifier reads "child_of" or "parent_of").

In [ ]:
parent_relations = pl.DataFrame({"child": ["c1", "c2", "c4"], "parent": ["s1", "s1", "c2"]})
show(resolve_transformations(RolesFromParents(qualifiers=["made_from"], direction="child_of"), candidates, e2hu, handling_units, events, processing, parents=parent_relations))

### Option 4: produced and consumed objects

The user names which objects are produced ($HU_{prod}$) and which are consumed ($HU_{cons}$), via object attribute values or a table.
prod(e) = produced objects whose first event is e; cons(e) = consumed objects whose last event is e (`consumption="final"`) or, with `"partial"`, also consumed objects that are in an event together with a produced one.
Here the sets are given directly.

In [ ]:
object_sets = (pl.Series("hu", ["s1", "c1", "c2", "c4"]), pl.Series("hu", ["s1", "c2"]))
option = RolesFromObjectSets(produced=None, consumed=None, consumption="final")
show(resolve_transformations(option, candidates, e2hu, handling_units, events, processing, object_sets=object_sets))

### Option 6: frequency of first and last activities

`activity_frequencies(e2hu, first_last)`: per object type and activity, how often the activity is the first (`prod`) or last (`cons`) event of a handling unit of that type, relative to all events of that type with that activity.
Activities with frequency ≥ $\tau$ make their events transformation events; T is derived.
With $\tau = 0.5$ on the running example, "quality check" (last event of 4 of 5 coils) also counts as consuming, so e14, e16 and e17 become transformation events: the option fits logs where the transformations are not otherwise known, and $\tau$ is the lever.

In [ ]:
activity_frequencies(e2hu, first_last_events(e2hu, processing)).sort("object_type", "ttype", "activity")

In [ ]:
show(resolve_transformations(RolesFromFrequency(threshold=0.5), None, e2hu, handling_units, events, processing))

## 4. Assumption checks

- `apply_as_t1(candidates, roles)`: AS-T1 (every transformation event consumes or produces something). Drops other candidates (notice "AS-T1").
- `check_assumptions(e2hu, roles)`: AS-T2 (a produced handling unit has no event before its producing event), AS-T3 (a fully consumed handling unit, i.e. consumed by an event that produces nothing, occurs in no later event), AS-T4 (no two transformation events at the same timestamp share a consumed or produced handling unit). Violations are notices; they do not block.

On the running example all hold; a synthetic violation of AS-T2: s1 (first event e1) produced again at e15.

In [ ]:
broken = pl.concat([resolved.roles, pl.DataFrame({"event": ["e15"], "hu": ["s1"], "role": ["produced"]})])
apply_as_t1(with_e4, resolved.roles)[2], check_assumptions(e2hu, broken)

## 5. Save

The resolved roles of the running example configuration, for later notebooks (e.g. `lineage.ipynb`).

In [ ]:
DERIVED.mkdir(exist_ok=True)
resolved.roles.sort("event", "role", "hu").write_csv(DERIVED / "transformation_roles.csv")